**# COMP20008 - Assignment 2**

**Research Question**:

To what extent can behavioural, spatial, temporal, and environmental features predict whether a squirrel is observed eating, and which factors are most influential for this behaviour?

## 1. Imports


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import datetime
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.metrics import normalized_mutual_info_score
from sklearn.metrics import (accuracy_score,
                            precision_score,
                            recall_score,
                            f1_score,
                            classification_report,
                            confusion_matrix,
                            ConfusionMatrixDisplay)



---



## 2. Load Raw Data

In [ ]:
squirrel_df = pd.read_csv('/content/squirrel.csv')
hectare_df = pd.read_csv('/content/hectare.csv')

print('squirrel.csv shape: ', squirrel_df.shape)
print('hectare.csv shape: ', hectare_df.shape)

In [ ]:
# missing value counts for squirrel.csv
missing_squirrel = squirrel_df.isnull().sum()
missing_pct_squirrel = round((missing_squirrel/len(squirrel_df)) * 100, 2)
missing_info_squirrel = pd.DataFrame({'Missing Count': missing_squirrel, 'Missing Percentage': missing_pct_squirrel})
missing_info_squirrel.loc[missing_info_squirrel['Missing Count']>0]

In [ ]:
# missing value counts for hectare.csv
missing_hectare = hectare_df.isnull().sum()
missing_pct_hectare = round((missing_hectare/len(hectare_df)) * 100, 2)
missing_info_hectare = pd.DataFrame({'Missing Count': missing_hectare, 'Missing Percentage': missing_pct_hectare})
missing_info_hectare.loc[missing_info_hectare['Missing Count']>0]

### Merging squirrel.csv and hectare.csv

We have decided to include and analyse 4 columns from the `hectare.csv`. These columns are related to eating behaviour and have low percentage of missing values. These include:


*   `Hectare Conditions`: How busy/calm the park area was at the time of sighting
*   `Total Time of Sighting`: This is the sighting session duration. Longer sightings give more oppotunities to record eating.
*   `Number of sighters`: How many sighters that observed the hectare for the sighitng session.
*    `Number of Squirrels`: Total number of squirrels observed in the hectare.

| Column/Feature | Justification for Removal |
| -------- | -------- |
|`Litter` | Has ~54% missing, too sparse to impute reliably |
| `Litter Notes` | Has ~99% missing, almost empty |
| `Hectare Conditions Notes` | Has ~89% missing, mixed text |
| `Other Animal Sighting` | text with no consistent format |
| `Sighter Observed Weather Data` | inconsistent entries, could not be reliably encoded without manual cleaning |
| `Anonymized Sighter` | Observer's identifier, no relation to squirrel eating behaviour |





In [ ]:
# merge squirrel.csv and hectare.csv for its chosen columns
# based on `Hectare` and `Shift` columns
hectare_columns = ['Hectare', 'Shift', 'Date', 'Hectare Conditions',
                   'Number of Squirrels', 'Number of sighters',
                   'Total Time of Sighting']
df = squirrel_df.merge(hectare_df[hectare_columns], on=['Hectare', 'Shift', 'Date'], how='left')
df.shape



---



## 3. Eating Behaviour Information


In [ ]:
# analyse the distribution of target feature
eating_counts = squirrel_df['Eating'].value_counts()
eating_pct    = squirrel_df['Eating'].value_counts() / eating_counts.sum() * 100

# combine into one dataframe
eating_distribution = pd.DataFrame(
    {'Count': eating_counts,
     'Percentage': eating_pct.round(1)})
print(eating_distribution)

In [ ]:
# visualization of the distribution
fig, ax = plt.subplots(figsize=(3.5, 3))
eating_pct.plot(kind='bar', ax=ax, color=['tab:blue', 'tab:orange'])

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', fontsize=8)

ax.set_title('Distribution of Eating', fontweight='bold')
ax.set_xticklabels(['Not Eating', 'Eating'], rotation=0)
ax.set_xlabel('')
ax.set_ylabel('Percentage (%)')
ax.set_ylim(0, 100)

plt.tight_layout()
plt.savefig('eating_distribution_percentage.png', dpi=300, bbox_inches='tight')
plt.show()



---



## 4. Dropping Irrelevant Features

Let's consider all the columns/features in the `squirrel.csv` file

In [ ]:
list(squirrel_df.columns)

We will remove columns/features of the data that will not assisst us in answering the research question.

Each column/feature that is removed is justified by an explanation.


| Column/Feature | Justification for Removal |
| ---------------| -------------------------|
| `Unique Squirrel ID` | Only a label for record keeping purposes &rarr; no predictive value |
| `Hectare Squirrel Number` | Number within the chronological squence of squirrels sighting &rarr; not a property of squirrel    |
| `Highlight Fur Color` | Has ~36% missing values |
| `Combination of Primary and Highlight Color` | It is a combiantion of `Primary Fur Color` and `Highlight Fur Color` columns|
| `Color notes` | Has ~94% missing values |
| `Specific Location` | Has ~84% missing values, already covered by `X` and `Y` columns|
| `Other Activities` | Has ~86% missing values|
| `Other Interactions` | Has ~92% missing values|
| `Lat/Long` | Redundant as `X` and `Y` will already cover it |


In [ ]:
columns_to_drop = ['Unique Squirrel ID',
                   'Hectare Squirrel Number',
                   'Highlight Fur Color',
                   'Combination of Primary and Highlight Color',
                   'Color notes',
                   'Specific Location',
                   'Other Activities',
                   'Other Interactions',
                   'Lat/Long']

In [ ]:
df = df.drop(columns=columns_to_drop)
print(f'Number of dataframe columns after dropping: {df.shape[1]}')

# columns left
print('Available columns for anlysis: ')
for col in df.columns:
  print('\t' + col)



---



## 5. Preprocessing



### Handling Missing Values





#### `Age`
This column include Nan and '?' as values.

'?' is most likely entered to communicate that the age of the squirrel was not identifiable by the observer. Therefore, instead of imputing with modal value or deleting rows, we decided to treat it as a separate category 'Unknown'.

For handling null values in age column, we imputed it with the modal value. This is because `Age` is categorical (either `Adult` or `Juvenile`). Therefore, using mean or median is useless.

In [ ]:
# `Age` column before
print('`Age` before imputing: ')
print(df['Age'].value_counts(dropna=False))

# impute `Age` column with modal value
age_mode = df['Age'].mode()[0]
# new 'Unknown` category for '?' values
df['Age'] = df['Age'].replace('?', 'Unknown')
# handle null values
df['Age'] = df['Age'].fillna(age_mode)

# `Age` column after
print('\n`Age` after imputing:')
print(df['Age'].value_counts())

#### `Primary Fur Color`
Similar to `Age` column, we imputed null values in this column with modal value.

In [ ]:
# `Primary Fur Color` column before
print('`Primary Fur Color` before imputing: ')
print(df['Primary Fur Color'].value_counts(dropna=False))

# impute `Primary Fur Color` column with modal value
fur_mode = df['Primary Fur Color'].mode()[0]
# handle null values
df['Primary Fur Color'] = df['Primary Fur Color'].fillna(fur_mode)

# `Age` column after
print('\n`Primary Fur Color` after imputing:')
print(df['Primary Fur Color'].value_counts())

#### `Location`
Similar to previous columns, imputed this column with modal value.

In [ ]:
# `Location` column before
print('`Location` before imputing: ')
print(df['Location'].value_counts(dropna=False))

# impute `Location` column with modal value
loc_mode = df['Location'].mode()[0]
# handle null values
df['Location'] = df['Location'].fillna(loc_mode)

# `Location` column after
print('\n`Location` after imputing:')
print(df['Location'].value_counts())

#### `Hectare Conditions`
Similar to previous columns, imputed this column with modal value.

In [ ]:
# `Hectare Conditions` column before
print('`Hectare Conditions` before imputing: ')
print(df['Hectare Conditions'].value_counts(dropna=False))

# 'Calm, Busy' and 'Medium' conditions are replaced with new category 'Moderate'
df['Hectare Conditions'] = df['Hectare Conditions'].replace({'Calm, Busy': 'Moderate',
                                                             'Medium': 'Moderate'})
# impute `Hectare Conditions` column with modal value
hectare_mode = df['Hectare Conditions'].mode()[0]
# handle null values
df['Hectare Conditions'] = df['Hectare Conditions'].fillna(hectare_mode)

# `Hectare Conditions` column after
print('\n`Hectare Conditions` after imputing:')
print(df['Hectare Conditions'].value_counts())

#### `Number of Squirrels`

In [ ]:
# `Number of Squirrels` column before
print('`Number of Squirrels` before imputing: ')
print(df['Number of Squirrels'].value_counts(dropna=False))

# impute `Number of Squirrels` column with median value
nsq_med = df['Number of Squirrels'].median()
# handle null values
df['Number of Squirrels'] = df['Number of Squirrels'].fillna(nsq_med)

# `Number of Squirrels` column after
print('\n`Number of Squirrels` after imputing:')
print(df['Number of Squirrels'].value_counts())

#### `Number of sighters`
Similar to previous columns, imputed this column with median value.

In [ ]:
# `Number of sighters` column before
print('`Number of sighters` before imputing: ')
print(df['Number of sighters'].value_counts(dropna=False))

# impute `Number of sighters` column with modal value
nsighters_med = df['Number of sighters'].median()
# handle null values
df['Number of sighters'] = df['Number of sighters'].fillna(nsighters_med)

# `Number of sighters` column after
print('\n`Number of sighters` after imputing:')
print(df['Number of sighters'].value_counts())

#### `Total Time of Sighting`
This is a continuos column that might include some outliers. Therefore, it is imputed with median.

In [ ]:
# `Total Time of Sighting` column before
print('`Total Time of Sighting` before imputing: ')
print(df['Total Time of Sighting'].value_counts(dropna=False))

# impute `Total Time of Sighting` column with median value
time_med = df['Total Time of Sighting'].median()
# handle null values
df['Total Time of Sighting'] = df['Total Time of Sighting'].fillna(time_med)

# `Total Time of Sighting` column after
print('\n`Total Time of Sighting` after imputing:')
print(df['Total Time of Sighting'].value_counts())

### Feature Engineering


#### `Above Ground Sighter Measurement`
This column measures the height above the ground where the squirrel was observed. It includes different types of data, both numeric and string, and contains nan values.

If `Above Ground Sighter Measurement` is False, then it means that squirrel sightings was on ground plane (no height difference from ground and squirrel)

We decided to treat null columns as False.

In [ ]:
def clean_above_ground(val):
  if pd.isna(val):
    return 0.0
  if str(val)=='FALSE':
    return 0.0
  else:
    return float(val)

df['Above Ground Height'] = df['Above Ground Sighter Measurement'].apply(clean_above_ground)

# don't need Above Ground Sigher Measurement
df = df.drop(columns=['Above Ground Sighter Measurement'])

#### `Is Weekend`
This column is added to the dataframe for further analysis. It has binary values (0 for weekdays and 1 for weekends)

In [ ]:
# Parse Date (stored as MMDDYYYY integer) and extract Is_weekend
df['Formatted Date'] = pd.to_datetime(df['Date'].astype(str), format='%m%d%Y')
df['Is Weekend'] = (df['Formatted Date'].dt.dayofweek >= 5).astype(int)

# Drop Date and the Formatted Date Column
df = df.drop(columns=['Date', 'Formatted Date'])

print('`Is Weekend` value counts:')
print(df['Is Weekend'].value_counts())

#### Check for Outliers

In [ ]:
# Inspect continuous features for extreme values before modelling
continuous_cols = ['X', 'Y', 'Above Ground Height', 'Number of Squirrels',
                   'Number of sighters', 'Total Time of Sighting']

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.flatten(), continuous_cols):
    ax.boxplot(df[col].dropna(), vert=True)
    ax.set_title(col, fontsize=10)
    ax.set_ylabel('Value')
plt.suptitle('Outlier Inspection: Continuous Features', fontweight='bold')
plt.tight_layout()
plt.savefig('outlier_inspection.png', dpi=150, bbox_inches='tight')
plt.show()

# IQR-based flag count
print('Outlier counts using 1.5 × IQR rule:')
for col in continuous_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f'  {col}: {n} flagged ({n/len(df)*100:.2f}%) | bounds: [{lower:.3f}, {upper:.3f}]')
plt.savefig('Outlier Inspection')

### Encode Categorical Values


In [ ]:
# drop Hectare column
df = df.drop(columns=['Hectare'])

In [ ]:
# for shift, map AM to 0 and PM to 1
def encode_shift(shift):
  if shift=='AM':
    return 0
  if shift=='PM':
    return 1
df['Shift'] = df['Shift'].apply(encode_shift)

In [ ]:
# for Location, map Ground Plane to 0, Above Ground 1
def encode_location(loc):
  if loc=='Ground Plane':
    return 0
  if loc=='Above Ground':
    return 1
df['Location'] = df['Location'].apply(encode_location)

In [ ]:
# for Age, one-hot-encoding
onehot_encoder = (OneHotEncoder(sparse_output=False).set_output(transform='pandas'))
encoded_age = onehot_encoder.fit_transform(df[['Age']]).astype(int)
df = pd.concat([df, encoded_age], axis=1)
df = df.drop(columns=['Age'])

In [ ]:
# for Primary Fur Color, one-hot-encoding
onehot_encoder = (OneHotEncoder(sparse_output=False).set_output(transform='pandas'))
encoded_fur_color = onehot_encoder.fit_transform(df[['Primary Fur Color']]).astype(int)
df = pd.concat([df, encoded_fur_color], axis=1)
df = df.drop(columns=['Primary Fur Color'])

In [ ]:
# for Hectare Conditions, one-hot-encoding
onehot_encoder = (OneHotEncoder(sparse_output=False).set_output(transform='pandas'))
encoded_hect_cond = onehot_encoder.fit_transform(df[['Hectare Conditions']]).astype(int)
df = pd.concat([df, encoded_hect_cond], axis=1)
df = df.drop(columns=['Hectare Conditions'])

### Convert Boolean Columns to Integer
Many columns (such as `Running`, `Climbing`, etc.) include values of True and False.

These are replaces with 0 for False and 1 for True.

In [ ]:
df.head()
df.columns

In [ ]:
bool_columns = ['Running', 'Chasing', 'Climbing', 'Eating', 'Foraging', 'Kuks', 'Quaas', 'Moans',
                'Tail flags', 'Tail twitches', 'Approaches', 'Indifferent', 'Runs from']
df[bool_columns] = df[bool_columns].astype(int)

After preprocessing, the data looks like:

In [ ]:
print(df.dtypes)

In [ ]:
print(df.isnull().sum())



---



## 6. Creating Feature List

Here, we will define specific sets of features to be analysed based on the research question.

In [ ]:
# behavioural features
behav_features = ['Running', 'Chasing', 'Climbing', 'Foraging',
                  'Kuks', 'Quaas', 'Moans', 'Tail flags', 'Tail twitches',
                  'Approaches', 'Indifferent', 'Runs from']
# spatial features
spat_features = ['X', 'Y', 'Location', 'Above Ground Height']

# temporal features
temp_features = ['Shift', 'Is Weekend']

# characteristic features
char_features = ['Age_Adult', 'Age_Juvenile', 'Age_Unknown',
                  'Primary Fur Color_Black', 'Primary Fur Color_Cinnamon',
                  'Primary Fur Color_Gray']
# environmental featues
env_features = ['Hectare Conditions_Busy', 'Hectare Conditions_Calm',
                'Hectare Conditions_Moderate', 'Number of Squirrels',
                'Number of sighters', 'Total Time of Sighting']

features = behav_features + spat_features + temp_features + char_features + env_features



---



## 7. Scaling Data

In [ ]:
numerical_cols = ['X', 'Y', 'Above Ground Height', 'Number of Squirrels',
                  'Number of sighters', 'Total Time of Sighting']

In [ ]:
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[numerical_cols] = scaler.fit_transform(df[numerical_cols])



---



## 8. Saving Data

In [ ]:
df.shape


In [ ]:
df.to_csv('preprocessed_squirrel.csv', index=False)
df_scaled.to_csv('preprocessed_squirrel_scaled.csv', index=False)



---



## 9. Correlation


### Pearson correlation

 Correlation between all selected features and Eating

In [ ]:
correlation_data = df.copy()
# Calculate Pearson correlation between each selected feature and Eating
correlations = correlation_data[features + ['Eating']].corr()

# Select only correlations with the target variable Eating
eating_corr = correlations['Eating'].drop('Eating')

# Sort by absolute correlation strength
eating_corr_sorted = eating_corr.reindex(
    eating_corr.abs().sort_values(ascending=False).index
)

print('Correlation between selected features and Eating:')
print(eating_corr_sorted)

Plot top 10 features correlated with Eating

In [ ]:
# better for report, takes less space
top10_corr = eating_corr_sorted.head(10)
fig, ax = plt.subplots(figsize=(6, 3.5))
top10_corr.plot(kind='barh', ax=ax)
ax.set_xlabel('Correlation with Eating', fontsize=9)
ax.set_ylabel('Feature', fontsize=9)
ax.set_title('Top 10 Features correlated with Eating', fontweight='bold', fontsize=10)
ax.tick_params(axis='both', labelsize=8)
ax.bar_label(ax.containers[0], fmt='%.3f', fontsize=7, padding=2)

# Add extra space on both sides of the x-axis
xmin, xmax = ax.get_xlim()
ax.set_xlim(xmin * 1.25, xmax * 1.25)

plt.tight_layout()
plt.savefig('Top10_correlations.png', bbox_inches='tight', dpi=200)
plt.show()

In [ ]:
top10_corr = eating_corr_sorted.head(10)
fig, ax = plt.subplots(figsize = (11, 6))
top10_corr.plot(kind = 'barh', ax = ax)
ax.set_xlabel('Correlation with Eating')
ax.set_ylabel('Feature')
ax.set_title('Top 10 Features correlated with Eating', fontweight='bold')
ax.bar_label(
    ax.containers[0],
    fmt = '%.3f', fontsize=8
)
plt.tight_layout()
plt.savefig('Top10_correlations.png', bbox_inches='tight', dpi=200)
plt.show()

In [ ]:
print('Top 10 strongest correlations with Eating:')
print(eating_corr_sorted.head(10))

Top positive correlations

In [ ]:
positive_corr = eating_corr[eating_corr > 0].sort_values(ascending = False).head(10)
fig, ax = plt.subplots(figsize=(10, 5))
positive_corr.plot(kind = 'barh', ax = ax)
ax.set_xlabel('Correlation with Eating')
ax.set_ylabel('Feature')
ax.set_title('Top Positive Correlations with Eating', fontweight='bold')
ax.bar_label(
    ax.containers[0],
    fmt = '%.3f', fontsize=8)
plt.tight_layout()
plt.savefig('Top10_positive_correlated.png')
plt.show()

In [ ]:
print('Top postive correlations with Eating:')
print(positive_corr)

In [ ]:
negative_corr = eating_corr[eating_corr < 0].sort_values(ascending = True).head(10)

fig, ax = plt.subplots(figsize = (12, 6))

negative_corr.plot(kind = 'barh', ax = ax)

ax.set_xlabel('Correlation with Eating')
ax.set_ylabel('Feature')
ax.set_title('Top Negative Correlations with Eating')

ax.bar_label(
    ax.containers[0],
    fmt = '%.3f',
    padding = 3
)

plt.tight_layout()
plt.savefig('Top10_negative_correlated.png')
plt.show()

In [ ]:
print('Top negative correlations with Eating:')
print(negative_corr)

Compare average correlations strength by feature groups

In [ ]:
# Define feature group
group_name = [
    'Behavioural',
    'Spatial',
    'Temporal',
    'Environment',
    'Characteristic'
]
group_features = [
    behav_features,
    spat_features,
    temp_features,
    env_features,
    char_features
]

# calculate each group absolute average value
group_corr_value = []

for feature_in_group in group_features:
  avg_corr = eating_corr[feature_in_group].abs().mean()
  group_corr_value.append(avg_corr)

group_corr = pd.Series(group_corr_value, index = group_name)
group_corr = group_corr.sort_values(ascending = False)
print(group_corr)


Plot average correlation strength by feature groups

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
group_corr.plot(kind='bar', ax=ax)
ax.set_xlabel('Feature Group', fontsize=9)
ax.set_ylabel('Average Correlation with Eating', fontsize=9)
ax.set_title('Average Correlation Strength by Feature Group', fontweight='bold', fontsize=10)
ax.tick_params(axis='both', labelsize=8)
ax.bar_label(ax.containers[0], fmt='%.3f', fontsize=7, padding=2)

# Add headroom so the top labels don't get cut off
ax.set_ylim(0, max(group_corr.values) * 1.15)

# Rotate x-tick labels if they're long
plt.xticks(rotation=30, ha='right')

plt.tight_layout()
plt.savefig('Groups_average_correlation.png', bbox_inches='tight', dpi=200)
plt.show()

In [ ]:
print('Behavioural Correlations:')
print(eating_corr[behav_features].sort_values())

print('\n Spatial Correlations:')
print(eating_corr[spat_features].sort_values())

print('\nTemporal Correlations:')
print(eating_corr[temp_features].sort_values())

print('\nEnvironment Correlations:')
print(eating_corr[env_features].sort_values())

print('\nCharacteristic Correlations:')
print(eating_corr[char_features].sort_values())

In [ ]:
# calculate postive and negative average correlations separately between each group

postive_counts = []
negative_counts = []

for feature_in_group in group_features:
  group_values = eating_corr[feature_in_group]
  postive_counts.append((group_values > 0).sum())
  negative_counts.append((group_values < 0).sum())

group_direction = pd.DataFrame({
    'Postive Count': postive_counts,
    'Negative Count': negative_counts
}, index = group_name)

print(group_direction)

In [ ]:
# Plot
fig, ax = plt.subplots(figsize = (12, 6))
group_direction.plot(kind = 'bar', ax = ax)
ax.set_xlabel('Feature Group')
ax.set_ylabel('Number of Features')
ax.set_title('Number of Positive and Negative Correlations by Feature Group')
for container in ax.containers:
    ax.bar_label(
        container,
        fmt = '%.0f',
        padding = 3
    )
plt.tight_layout()
plt.savefig('Number_in_groups.png')
plt.show()

Since Pearson correlation between features and groups are quiet samll, so there just weak linear relationship. So we try to another way to find the correlation.

###  Mutual Information

In [ ]:
mi_data = df.copy()
X = mi_data[features].copy()
Y = mi_data['Eating']

### K-bins

In [ ]:
# Continuous features need to discretisation befor MI
continuous_features = [
    'X',
    'Y'
]
continuous_features = [
    feature for feature in continuous_features
    if feature in X.columns
]
discretiser = KBinsDiscretizer(
    n_bins = 3,
    encode = 'ordinal',
    strategy = 'quantile'
)
X_discrete = X.copy()
X_discrete[continuous_features] = discretiser.fit_transform(
    X[continuous_features]
)

# Calculate MI
mi_score = mutual_info_classif(
    X_discrete,
    Y,
    discrete_features = True,
    random_state = 42
)
mi_result = pd.Series(
    mi_score,
    index = X_discrete.columns
)
mi_result = mi_result.sort_values(ascending = False)


In [ ]:
print('Features by MI')
print(mi_result)

In [ ]:
top10_mi = mi_result.head(10)
fig, ax = plt.subplots(figsize = (12, 6))
top10_mi.plot(kind = 'barh', ax = ax)
ax.set_xlabel('MI Score')
ax.set_ylabel('Feature')
ax.set_title('Top 10 Features by MI')
ax.bar_label(
    ax.containers[0],
    fmt = '%.3f',
    padding = 3
)
plt.subplots_adjust(left = 0.35)
plt.tight_layout()
plt.savefig('Top10_features_MI.png')
plt.show()

In [ ]:
# better for report, takes less space
top_mi = mi_result.head(8).sort_values()

fig, ax = plt.subplots(figsize=(5.5, 3.2))

top_mi.plot(kind='barh', ax=ax)

ax.set_title('Top 10 Mutual Information Scores', fontweight='bold')
ax.set_xlabel('Mutual Information')
ax.set_ylabel('')

ax.bar_label(
    ax.containers[0],
    fmt = '%.3f',
    padding = 3
)

plt.tight_layout()
plt.savefig('top_mi_scores.png', dpi=300, bbox_inches='tight')
plt.show()

### Compare average MI by feature group

In [ ]:
group_mi_value = []
for feature_in_group in group_features:
  valid_features = []
  for feature in feature_in_group:
    if feature in mi_result.index:
      valid_features.append(feature)
  avg_mi = mi_result[valid_features].mean()
  group_mi_value.append(avg_mi)

group_mi = pd.Series(group_mi_value, index = group_name)
group_mi = group_mi.sort_values(ascending = False)
print('Average Mi by feature group:')
print(group_mi)


In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
group_mi.plot(kind='bar', ax=ax)
ax.set_xlabel('Feature Group', fontsize=9)
ax.set_ylabel('Average MI', fontsize=9)
ax.set_title('Average MI by Feature Group', fontweight='bold', fontsize=10)
ax.tick_params(axis='both', labelsize=8)
ax.bar_label(ax.containers[0], fmt='%.4f', fontsize=7, padding=2)

# Add headroom so top labels aren't cut off
ax.set_ylim(0, max(group_mi.values) * 1.15)

# Rotate x-tick labels for readability
plt.xticks(rotation=30, ha='right')

plt.tight_layout()
plt.savefig('Groups_average_MI.png', bbox_inches='tight', dpi=200)
plt.show()

### Normalized Mutual Information

In [ ]:
nmi_value = []
# Calculate NMI between each feature and Eating
for feature in X_discrete.columns:
  nmi = normalized_mutual_info_score(X_discrete[feature], Y)
  nmi_value.append(nmi)

nmi_result = pd.Series(nmi_value, index = X_discrete.columns)
nmi_result = nmi_result.sort_values(ascending = False)

print('Features by NMI')
print(nmi_result)

In [ ]:
top10_nmi = nmi_result.head(10)
fig, ax = plt.subplots(figsize=(6, 3.5))
top10_nmi.plot(kind='barh', ax=ax)
ax.set_xlabel('NMI Score', fontsize=9)
ax.set_ylabel('Feature', fontsize=9)
ax.set_title('Top 10 Features by NMI', fontweight='bold', fontsize=10)
ax.tick_params(axis='both', labelsize=8)
ax.bar_label(ax.containers[0], fmt='%.3f', fontsize=7, padding=2)

# Add headroom on the right so labels don't get clipped
xmin, xmax = ax.get_xlim()
ax.set_xlim(xmin, xmax * 1.15)

plt.tight_layout()
plt.savefig('Top10_features_NMI.png', bbox_inches='tight', dpi=200)
plt.show()

In [ ]:
group_nmi_value = []
for feature_group in group_features:
    valid_features = []
    for feature in feature_group:
        if feature in nmi_result.index:
            valid_features.append(feature)
    avg_nmi = nmi_result[valid_features].mean()
    group_nmi_value.append(avg_nmi)
group_nmi = pd.Series(
    group_nmi_value,
    index = group_name
)
group_nmi = group_nmi.sort_values(ascending = False)
print('Average NMI by feature group:')
print(group_nmi)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
group_nmi.plot(kind='bar', ax=ax)
ax.set_xlabel('Feature Group', fontsize=9)
ax.set_ylabel('Average NMI', fontsize=9)
ax.set_title('Average NMI by Feature Group', fontweight='bold', fontsize=10)
ax.tick_params(axis='both', labelsize=8)
ax.bar_label(ax.containers[0], fmt='%.4f', fontsize=7, padding=2)

# Add headroom so top labels aren't cut off
ax.set_ylim(0, max(group_nmi.values) * 1.15)

# Rotate x-tick labels for readability
plt.xticks(rotation=30, ha='right')

plt.tight_layout()
plt.savefig('Groups_average_NMI.png', bbox_inches='tight', dpi=200)
plt.show()

### Feature-feature correlation

Check whether any two predictor features are highly correlated

In [ ]:
selected_feature = [
    'Running', 'Chasing', 'Climbing', 'Foraging',
    'Approaches', 'Indifferent', 'Runs from', 'Location',
    'Above Ground Height', 'Shift'
]

selected_feature_corr = df_scaled[selected_feature].corr()
print(selected_feature_corr)

In [ ]:
plt.figure(figsize = (6, 5))
sns.heatmap(
    selected_feature_corr,
    annot = True,
    cmap = 'coolwarm',
    center = 0,
    fmt = '.2f',
    annot_kws={'size': 6},
    cbar_kws={'shrink': 0.7}
)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.title('Correlations between selected predictor features', fontweight='bold')
plt.tight_layout()
plt.savefig('Feature_feature_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
nmi_heatmap_df = df[selected_feature].copy()
selected_feature_nmi = pd.DataFrame(
    index = selected_feature,
    columns = selected_feature,
    dtype = float
)
for feature_1 in selected_feature:
    for feature_2 in selected_feature:
        selected_feature_nmi.loc[feature_1, feature_2] = normalized_mutual_info_score(
            nmi_heatmap_df[feature_1],
            nmi_heatmap_df[feature_2]
        )
print(selected_feature_nmi)

In [ ]:
plt.figure(figsize = (6, 5))
sns.heatmap(
    selected_feature_nmi,
    annot = True,
    cmap = 'Blues',
    fmt = '.2f',
    annot_kws={'size': 6},
    cbar_kws={'shrink': 0.7}
)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.title('NMI between selected predictor features',
          fontweight='bold')
plt.tight_layout()
plt.savefig('Feature_feature_NMI_heatmap.png', dpi = 300, bbox_inches = 'tight')
plt.show()

As an addition check before modelling, feature-feature corrleations were examined to identify possible overlapping information among predictors, so that highly related variables could be considered carefully when interpreting feature importance.

#  Supervised Learning Models and Evaluation

## Define Features and Target

In [ ]:
# select feature as predictors
X = df.drop(columns=['Eating'])

# `Eating` as the target variable
y = df['Eating']

print('Number of features:', X.shape[1])

print('\nFeatures: ')
for feature in X.columns:
  print(f'\t{feature}')

## Train / Validation / Test Split

We split the data into training (70%), validation (10%), and test (20%) sets using the stratified sampling to preserve the eating imbalance across all three sets.

The test set is not looked at by the model until the final evaluation and tuning of hyperparameters.


In [ ]:
# first split to get test and train
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)

# second split to get test and validation
X_train, X_val, y_train, y_val = train_test_split(X_train,
                                                  y_train,
                                                  test_size=0.125,
                                                  random_state=42,
                                                  stratify=y_train)
# train/validation/test size
print('Training size:  ', X_train.shape[0])
print('Validation size: ', X_val.shape[0])
print('Test size:       ', X_test.shape[0])
print('\n')
# train/validation/test distribution
print('Proportion of "Eating" in train:     ', y_train.mean().round(3))
print('Proportion of "Eating" in validation:', y_val.mean().round(3))
print('Proportion of "Eating" in test:      ', y_test.mean().round(3))

## Decision Tree

### Tune Decision Tree `max_depth`

In [ ]:
depth_values = range(1, 20)

tree_tuning_scores = []

for depth in depth_values:
  tree = DecisionTreeClassifier(criterion='entropy',
                                max_depth=depth,
                                class_weight='balanced',
                                random_state=42)
  tree.fit(X_train, y_train)

  train_pred = tree.predict(X_train)
  val_pred = tree.predict(X_val)

  tree_tuning_scores.append({
        'max_depth': depth,
        'train_accuracy': accuracy_score(y_train, train_pred),
        'val_accuracy': accuracy_score(y_val, val_pred),
        'train_precision': precision_score(y_train, train_pred),
        'val_precision': precision_score(y_val, val_pred),
        'train_recall': recall_score(y_train, train_pred),
        'val_recall': recall_score(y_val, val_pred),
        'train_f1': f1_score(y_train, train_pred),
        'val_f1': f1_score(y_val, val_pred)
    })
tree_tuning_df = pd.DataFrame(tree_tuning_scores).round(3)
tree_tuning_df

#### Plot Decision Tree Tuning Result

In [ ]:
plot_tree_df = tree_tuning_df.copy()
plot_tree_df['max_depth_label'] = plot_tree_df['max_depth'].astype(str)

plt.plot(plot_tree_df['max_depth_label'],
         plot_tree_df['train_f1'],
         marker='o',
         label='Training F1-score')

plt.plot(plot_tree_df['max_depth_label'],
         plot_tree_df['val_f1'],
         marker='o',
         label='Validation F1-score')

plt.xlabel('Maximum Tree Depth')
plt.ylabel('F1-score')
plt.title('Decision Tree max_depth Tuning', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('Decision Tree max_depth Tuning')
plt.show()


#### Select Best `max_depth`

We tune max_depth using the validation set, selecting the value that maximises F1-score. The reason for choosing this score instead of accuracy is because of the imbalanced dataset.

For example a model that predicts 'not eating' for all records would achieve accuracy of ~0.75, however it is completly uninformative.

In [ ]:
best_tree_row = tree_tuning_df.loc[tree_tuning_df['val_f1'].idxmax()]

best_depth = best_tree_row['max_depth']

best_depth = int(best_depth)

print('Best max_depth:', best_depth)
print(best_tree_row)

### Final Decision Tree on Training and Validation Data

In [ ]:
final_tree = DecisionTreeClassifier(max_depth=best_depth,
                                    criterion='entropy',
                                    class_weight='balanced',
                                    random_state=42)
final_tree.fit(pd.concat([X_train, X_val], axis=0),
               pd.concat([y_train, y_val], axis=0))

final_tree_pred = final_tree.predict(X_test)

In [ ]:
plt.figure(figsize=(7, 3.8))
plot_tree(
    final_tree,                        # your trained tree — still depth 15
    feature_names=X_train.columns,
    class_names=['Not Eating', 'Eating'],
    filled=True,
    rounded=True,
    max_depth=2,                       # only show top 3 levels visually
    fontsize=6
    )
plt.title('Decision Tree — Top 2 Levels (full depth = 15)', fontweight='bold')
plt.tight_layout()
plt.savefig('Decision Tree', dpi=300, bbox_inches='tight')
plt.show()

## KNN

### Scale Data for KNN

StandardScaler is applied to the data for knn model. This is because knn is based on distances.

In [ ]:
scaler = StandardScaler()

# fit scaler on training data
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

### Tune KNN `n_neighbors`

We tune `k` using the `validation set`. k=1 is excluded as it causes overfitting. k=1 means that the model for each point, looks at its closest point and predicts according to that. This gives a model that is perfect for training and validation sets but not so much for unseen data.

In [ ]:
# only odd k values to avoid ties during voting
k_values = range(3, 33, 2)

knn_tuning_scores = []

for k in k_values:
  # build model with chosen k
  knn_model = KNeighborsClassifier(n_neighbors=k)

  knn_model.fit(X_train_scaled, y_train)

  train_pred = knn_model.predict(X_train_scaled)
  val_pred = knn_model.predict(X_val_scaled)

  # tuning dictionary
  knn_tuning_scores.append({
      'n_neighbours': k,
      'train_accuracy': accuracy_score(y_train, train_pred),
      'val_accuracy': accuracy_score(y_val, val_pred),
      'train_precision': precision_score(y_train, train_pred),
      'val_precision': precision_score(y_val, val_pred),
      'train_recall': recall_score(y_train, train_pred),
      'val_recall': recall_score(y_val, val_pred),
      'train_f1': f1_score(y_train, train_pred),
      'val_f1': f1_score(y_val, val_pred)
    })

knn_tuning_df = pd.DataFrame(knn_tuning_scores).round(3)
knn_tuning_df

#### Plot KNN Tuning Result

In [ ]:
plt.plot(knn_tuning_df['n_neighbours'],
         knn_tuning_df['train_f1'],
         marker='o',
         label='Training F1-score')

plt.plot(knn_tuning_df['n_neighbours'],
         knn_tuning_df['val_f1'],
         marker='o',
         label='Validation F1-score')

plt.xlabel('Number of Neighbours, k')
plt.ylabel('F1-score')
plt.title('KNN n_neighbours Tuning', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('KNN n_neighbors Tuning')
plt.show()


#### Select Best `k`

In [ ]:
best_knn_row = knn_tuning_df.loc[knn_tuning_df['val_f1'].idxmax()]
best_k = int(best_knn_row['n_neighbours'])

print('Best k:', best_k)
print(best_knn_row)

### Final KNN Model

In [ ]:
# combine training and validation data
X_train_val = pd.concat([X_train, X_val], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)

final_scaler = StandardScaler()

X_train_val_scaled = final_scaler.fit_transform(X_train_val)
X_test_scaled = final_scaler.transform(X_test)

# train finall KNN with best_k
final_knn = KNeighborsClassifier(n_neighbors=best_k)

final_knn.fit(X_train_val_scaled, y_train_val)

# evaluate on test set
final_knn_pred = final_knn.predict(X_test_scaled)

## Comparing Models

### Performance

In [ ]:
# function that gives the score for performance metrics
def model_performance(model_name, y_test, y_pred):
  accuracy = round(accuracy_score(y_test, y_pred), 3)
  precision = round(precision_score(y_test, y_pred, zero_division=0), 3)
  recall = round(recall_score(y_test, y_pred, zero_division=0), 3)
  f1 = round(f1_score(y_test, y_pred, zero_division=0), 3)

  metrics_dict = {'Model': model_name,
                  'Accuracy': accuracy,
                  'Precision': precision,
                  'Recall': recall,
                  'F1-score': f1}
  return metrics_dict
performance_results = []

# Baseline model: always predict the majority class, Not Eating
baseline_pred = [0] * len(y_test)

performance_results.append(
    model_performance('Baseline', y_test, baseline_pred)
)

performance_results.append(
    model_performance('Decision Tree', y_test, final_tree_pred)
)

performance_results.append(
    model_performance('KNN', y_test, final_knn_pred)
)

performance_df = pd.DataFrame(performance_results)

performance_df

### Plot Performances

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-score']

plot_df = performance_df.set_index('Model')[metrics].T

ax = plot_df.plot(kind='bar', width=0.65, figsize = (7,4))

for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=3, fontsize = 8)

plt.xlabel('Metric')
plt.ylabel('Score')
plt.title('Model Performance Comparison', fontweight='bold')
plt.xticks(rotation=0)
plt.legend(title='Model', loc='upper right')
plt.ylim(0, 0.9)
plt.tight_layout()
plt.savefig('Model_Performance_Comparison.png', dpi=300, bbox_inches='tight')
plt.show()

### Classification Reports

In [ ]:
print('Classification Reports')
print('-----------------------------------------------------')
print('Decision Tree Classification Report:')
print(classification_report(y_test,
                            final_tree_pred,
                            target_names=['Not Eating', 'Eating']))

print('-----------------------------------------------------')

print('KNN Classification Report:')
print(classification_report(y_test,
                            final_knn_pred,
                            target_names=['Not Eating', 'Eating']))

### Confusion Matrices

In [ ]:
# baseline confusion matrix
baseline_cm = confusion_matrix(y_test,
                               baseline_pred,
                               labels=[0, 1])
baseline_cm

In [ ]:
# decision tree confusion matrix
tree_cm = confusion_matrix(y_test,
                           final_tree_pred,
                           labels=[0,1])
tree_cm

In [ ]:
# knn confusion matrix
knn_cm = confusion_matrix(y_test,
                          final_knn_pred,
                          labels=[0, 1])
knn_cm

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

# Baseline
baseline_disp = ConfusionMatrixDisplay(
    confusion_matrix=baseline_cm,
    display_labels=['Not Eating', 'Eating']
)
baseline_disp.plot(ax=ax[0], cmap='Blues', colorbar=False)
ax[0].set_title('Baseline')


# Decision Tree
tree_disp = ConfusionMatrixDisplay(confusion_matrix=tree_cm,
                                   display_labels=['Not Eating', 'Eating'])

tree_disp.plot(ax=ax[1], cmap='Blues',
               colorbar=False)

ax[1].set_title('Decision Tree')

# KNN
knn_disp = ConfusionMatrixDisplay(confusion_matrix=knn_cm,
                                  display_labels=['Not Eating', 'Eating'])

knn_disp.plot(ax=ax[2], cmap='Blues', colorbar=False)
ax[2].set_title('KNN')

plt.suptitle('Confusion Matrices — Baseline vs Decision Tree vs KNN',
             fontweight='bold')
plt.tight_layout()
plt.savefig('Confusion_Matrices.png', dpi=300, bbox_inches='tight')
plt.show()

## Feature Importance

### Decision Tree and KNN Feature Importance

In [ ]:
tree_importance = pd.DataFrame({'Feature': X.columns,
                                'Importance': final_tree.feature_importances_},).round(3)
tree_importance = tree_importance.sort_values(
    by='Importance',
    ascending=False
)
top_tree = tree_importance.head(10).sort_values(by='Importance')

In [ ]:
knn_perm = permutation_importance(final_knn,
                                  X_test_scaled,
                                  y_test,
                                  scoring='f1',
                                  n_repeats=10,
                                  random_state=42)

knn_importance = pd.DataFrame({'Feature': X.columns,
                               'Importance': knn_perm.importances_mean})

knn_importance = knn_importance.sort_values(by='Importance',
                                            ascending=False)

top_knn = knn_importance.head(10).sort_values('Importance')
knn_importance.head(10)

In [ ]:
# Combined feature importance figure for report
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Decision Tree feature importance
axes[0].barh(top_tree['Feature'], top_tree['Importance'])
axes[0].set_title('Decision Tree Feature Importance', fontweight='bold')
axes[0].set_xlabel('Feature Importance')
axes[0].set_ylabel('Feature')

# KNN permutation importance
axes[1].barh(top_knn['Feature'], top_knn['Importance']
              , color='tab:orange')
axes[1].set_title('KNN Permutation Importance', fontweight='bold')
axes[1].set_xlabel('Permutation Importance')
axes[1].set_ylabel('Feature')

plt.suptitle('Top Feature Importances: Decision Tree and KNN', fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
feature_comparison_df = pd.DataFrame({
    'Decision Tree Top Features': tree_importance['Feature'].head(10).values,
    'KNN Top Features': knn_importance['Feature'].head(10).values
})

feature_comparison_df

In [ ]:
tree_top_features = set(tree_importance['Feature'].head(10))
knn_top_features = set(knn_importance['Feature'].head(10))

common_features = tree_top_features.intersection(knn_top_features)

print("Common important features:")
print(common_features)

# Clustering

In [ ]:
# use scaled data
s_squirrel_df = pd.read_csv('/content/preprocessed_squirrel_scaled.csv')

## First cluster approach will be on the behaviours of squirrels

In [ ]:
behav_df = s_squirrel_df[behav_features].copy()

# Add a new column sounds that is 0 if 0 is recorded for 'Kuks', 'Quaas'
# and 'Moans'. Record 1 otherwise
Sounds = []
Sounds = (behav_df['Kuks'] + behav_df['Quaas'] + behav_df['Moans'] > 0).astype(int)
behav_df.loc[:,'Sounds'] = Sounds

# Remove the unwanted columns
behav_df = behav_df.drop(columns = ['Kuks', 'Quaas', 'Moans'])

### 1. The Elbow Method


In [ ]:
distortions = []
k_values = range(1, 11)

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=16)
    kmeans.fit(behav_df)
    distortions.append(kmeans.inertia_)

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(k_values, distortions, color='tab:blue', marker='x', linestyle='-')

ax.set_xlabel('k (Number of Clusters)', fontsize=9)
ax.set_ylabel('Distortion (Sum Of Squared Errors)', fontsize=9)
ax.set_title('The Elbow Method', fontweight='bold', fontsize=10)
ax.tick_params(axis='both', labelsize=8)

plt.tight_layout()
plt.savefig('cluster_approach_1_elbow.png', bbox_inches='tight', dpi=200)
plt.show()
print("The elbow point is identified to be k=4.")

### 2. Analyse behaviours within each cluster

In [ ]:
# run k-means with optimal number of clusters
clusters = KMeans(n_clusters = 4, random_state = 16)
clusters.fit(behav_df)

# cluster labels assigned to each row
cluster_labels = clusters.labels_

# for each cluster, compute the average behavioural values
cluster_means = behav_df.groupby(cluster_labels).mean()

# plot the 3 most likely and unlikely beahviours per cluster
for cluster_num in cluster_means.index:

  # sort beavhiours in decending order
  b_sorted_vals = cluster_means.loc[cluster_num].sort_values(ascending = False)

  likely = b_sorted_vals.head(3)
  unlikely = b_sorted_vals.tail(3)

  fig,(ax1,ax2) = plt.subplots(1,2, figsize=(7, 4.5), layout= 'constrained')

  # chart for likely behaviours
  bars1 = ax1.bar(likely.index, likely.values * 100)
  ax1.set_title(f'Most Exhibited Behaviours', fontsize = 11)
  ax1.set_xlabel('Behaviours')
  ax1.set_ylabel('Percentage of squirrels(%)')
  ax1.set_ylim(0,max(likely)* 100 * 1.05)
  # add a label on top of each bar
  ax1.bar_label(bars1, fmt='%d')

  # chart for unlikely behaviours
  bars2 = ax2.bar(unlikely.index, unlikely.values * 100, color = 'tab:orange')
  ax2.set_title(f'Least Exhibited Behaviours', fontsize = 11)
  ax2.set_xlabel('Behaviours')
  ax2.set_ylabel('Percentage of squirrels(%)')
  ax2.set_ylim(0,max(likely)* 100* 1.05)
  # add a label on top of each bar
  ax2.bar_label(bars2, fmt='%d')

  plt.suptitle(f'Behaviours for Cluster {cluster_num}', fontweight='bold')
  plt.savefig(f'cluster{cluster_num}_behaviour.png')


### 3. Compare the behaviour profiles per cluster with eating

In [ ]:
# compute mean earing rate per cluster
eating_rates_b = s_squirrel_df.groupby(cluster_labels)['Eating'].mean()

cluster_nums = ['Cluster 0', 'Cluster 1', 'Cluster 2', 'Cluster 3']

# create a dataframe
b_eating_df = pd.DataFrame({'Cluster': cluster_nums, 'Eating(%)': eating_rates_b.values * 100})

# add a difference from the average
average = 25.1
b_eating_df['Difference from Average (%)'] = b_eating_df['Eating(%)'] - average
b_eating_df = b_eating_df.round(1)

b_eating_df

### 4. Plot these clusters against location

In [ ]:
# colors for the clusters
colours = ['tab:blue', 'tab:pink', 'tab:orange', 'tab:green']

fig, ax = plt.subplots(layout = 'constrained')

# loop over each cluster and plot its points
for cluster_num in range(4):

  colour = colours[cluster_num]

  # collect rows belonging to this cluster
  in_cluster = cluster_labels == cluster_num

  # plot the points
  ax.scatter(squirrel_df['X'][in_cluster], squirrel_df['Y'][in_cluster],
  c = colour, label = f'Cluster {cluster_num}', s = 10, alpha = 0.8, edgecolors = 'none' )

# add labels to the graph
ax.legend()
ax.grid(True)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('Clusters of Squirrel Behavioral Features')

# save the graph
plt.savefig('b_clusters_against_location.png')

## Second cluster approach based on temporal & some environmental features

In [ ]:
# temporal and select environmental features
te_features = ['Shift', 'Is Weekend', 'Hectare Conditions_Calm', 'Hectare Conditions_Moderate', 'Hectare Conditions_Busy']
te_df = s_squirrel_df[te_features]

# rename some columns so their names are shorter
te_df = te_df.rename(columns={'Hectare Conditions_Calm': 'HC_Calm', 'Hectare Conditions_Moderate':'HC_Moderate', 'Hectare Conditions_Busy':'HC_Busy'})

### 1. The Elbow Method

In [ ]:
# store the sum of squared errors value for each k value
distortions = []
k_values = range(1,11)

# run k means for integers 1 to 10
for k in k_values:

  # set a seed so that results are reproducible
  kmeans = KMeans(n_clusters = k, random_state = 17)

  # fit the normalised encoded dataframe
  kmeans.fit(te_df)

  # find the sum of squared distance from each point to its centroid
  distortions.append(kmeans.inertia_)

# plot the line for Sum of Squared errors
fig, ax = plt.subplots()
ax.plot(k_values, distortions, 'bx-')

# set labels
ax.set_xlabel('k (Number of Clusters)')
ax.set_ylabel('Distortion (Sum Of Squared Errors)')
ax.set_title('The Elbow Method')

# save the graph
plt.savefig('cluster_approach_2_elbow.png') ## different name...
print("The elbow point is identified to be k=4.")

In [ ]:
# better for report, less white space
distortions = []
k_values = range(1, 11)

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=17)
    kmeans.fit(te_df)
    distortions.append(kmeans.inertia_)

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(k_values, distortions, color='tab:blue', marker='x', linestyle='-')

ax.set_xlabel('k (Number of Clusters)', fontsize=9)
ax.set_ylabel('Distortion (Sum Of Squared Errors)', fontsize=9)
ax.set_title('The Elbow Method', fontweight='bold', fontsize=10)
ax.tick_params(axis='both', labelsize=8)

plt.tight_layout()
plt.savefig('cluster_approach_2_elbow.png', bbox_inches='tight', dpi=200)
plt.show()
print("The elbow point is identified to be k=4.")

### 2. Analyse the features within each cluster

In [ ]:
# run k-means with optimal number of clusters
clusters = KMeans(n_clusters=4, random_state=17)
clusters.fit(te_df)

# cluster labels assigned to each row
cluster_labels = clusters.labels_

# for each cluster, compute the average behavioural values
cluster_means = te_df.groupby(cluster_labels).mean()

for cluster_num in cluster_means.index:

  # sort beavhiours in decending order
  te_sorted_vals = cluster_means.loc[cluster_num].sort_values(ascending = False)

  fig,ax = plt.subplots(layout= 'constrained')

  bars = ax.bar(te_sorted_vals.index, te_sorted_vals.values * 100)

  # add labels
  ax.set_title(f'Features Exhibited in Cluster {cluster_num}', fontsize = 11)
  ax.set_ylabel('Percentage of squirrels(%)')
  ax.set_xlabel('Features')
  ax.set_ylim(0,max(te_sorted_vals.values * 100)* 1.05)
  # add a label on top of each bar
  ax.bar_label(bars, fmt='%d')

  plt.savefig(f'cluster{cluster_num}_te_features.png')


In [ ]:
# better for report, less white space
for cluster_num in cluster_means.index:

    te_sorted_vals = cluster_means.loc[cluster_num].sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(5, 3), constrained_layout=True)

    # colour bars by value — high is tab:blue, zero is tab:orange
    colors = ['tab:blue' if v > 0 else 'tab:orange' for v in te_sorted_vals.values]
    bars = ax.bar(te_sorted_vals.index, te_sorted_vals.values * 100, color=colors)

    ax.set_title(f'Features Exhibited in Cluster {cluster_num}', fontweight='bold', fontsize=10)
    ax.set_ylabel('Percentage of squirrels (%)', fontsize=8)
    ax.set_xlabel('Features', fontsize=8)
    ax.set_ylim(0, max(te_sorted_vals.values * 100) * 1.15)
    ax.tick_params(axis='both', labelsize=7)
    ax.bar_label(bars, fmt='%d', fontsize=7, padding=2)


    plt.savefig(f'cluster{cluster_num}_te_features.png', bbox_inches='tight', dpi=200)
    plt.show()

### 3. Eating rates per cluster

In [ ]:
# compute mean earing rate per cluster
te_eating_rates = s_squirrel_df.groupby(cluster_labels)['Eating'].mean()

cluster_nums = ['Cluster 0', 'Cluster 1', 'Cluster 2', 'Cluster 3']

# create a dataframe
te_eating_df = pd.DataFrame({'Cluster': cluster_nums, 'Eating(%)': te_eating_rates.values * 100})

# add a difference from the average
average = 25.1
te_eating_df['Difference from Average (%)'] = te_eating_df['Eating(%)'] - average
te_eating_df = te_eating_df.round(1)

te_eating_df

### 4. Plot these clusters against location

In [ ]:
# colors for the clusters
colours = ['tab:blue', 'tab:pink', 'tab:orange', 'tab:green']

fig, ax = plt.subplots(layout = 'constrained')

# loop over each cluster and plot its points
for cluster_num in range(4):

  colour = colours[cluster_num]

  # collect rows belonging to this cluster
  in_cluster = cluster_labels == cluster_num

  # plot the point
  ax.scatter(squirrel_df['X'][in_cluster], squirrel_df['Y'][in_cluster],
             c = colour, label = f'Cluster {cluster_num}', s =5)

# set labels
ax.legend()
ax.grid(True)
ax.set_title('Location of Temporal and Environmental Clusters')
ax.set_xlabel('X coordinate')
ax.set_ylabel('Y coordinate')

plt.savefig('te_clusters_against_location.png')
